# Chapter 3 — VICE: the same model, fewer approximations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/chemical-evolution-tutorial/blob/main/notebooks/03_vice.ipynb)

**Goal.** VICE (Johnson & Weinberg 2020, https://ui.adsabs.harvard.edu/abs/2020MNRAS.498.1364J) is a fast,
well-tested code for one-zone and multi-zone chemical evolution. We first make VICE solve *exactly* the model of
Chapter 2 and check that it gives the same answer. Then we switch on, one at a time, the physics our numpy model
left out, and measure what each changes. Finally we add bursts of star formation and run H21's models of the five
dwarfs.

**You will learn**
1. how to set up and run `vice.singlezone`;
2. how to validate a code against your own: same physics first, then one change at a time;
3. which approximations matter, and by how much;
4. how a burst of star formation makes the [α/Fe] rise seen in the LMC;
5. why [Fe/H] is not a clock: the five galaxies reach the same [Fe/H] at very different times.

**Run time.** On Colab, the first cell builds VICE from source, which takes about 8 minutes. Locally, install it
once with the same two commands.

In [ ]:
try:
    import vice
except ImportError:
    # VICE 1.3.1 is published only as source code, and its build fails with Cython >= 3.1. So: install an older Cython
    # first, then build VICE without pip's isolated build environment (which would fetch the newest Cython).
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cython>=3.0,<3.1', 'setuptools'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'vice==1.3.1'], check=True)
    import vice
print('VICE', vice.__version__)

In [ ]:
import hashlib, json, os, tempfile, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
DATA_URL = 'https://github.com/vasilybelokurov/chemical-evolution-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fetch(name, cache=Path('data')):
    """Return the local path of data file `name`, downloading it if needed; its SHA-256 must match the manifest."""
    want = MANIFEST['files'][name]['sha256']
    path = Path('../data_build') / name
    if not path.exists():
        path = cache / name
        if not path.exists() or sha256(path) != want:
            cache.mkdir(exist_ok=True)
            print(f'downloading {name} ...')
            urllib.request.urlretrieve(DATA_URL + name, path)
    assert sha256(path) == want, f'{name}: checksum mismatch'
    return path.resolve()


# Solar scale and atomic masses of Chapter 2
LOGEPS = {'fe': 7.47, 'si': 7.54, 'mg': 7.55}
A = {'fe': 56, 'si': 28, 'mg': 24}
X_H = 0.73
Y = np.load(fetch('h21_ssp_bundle.npz'))
i002 = min(len(Y['z']) - 1, int(np.searchsorted(Y['z'], 0.002, side='right')))
YIELD = {el: (float(Y[f'cc_{el}'][i002]), float(Y[f'ia_{el}'][i002])) for el in ('mg', 'si', 'fe')}

## 3.1 The numpy model of Chapter 2, again

For the comparison we need Chapter 2's model. It is copied here unchanged (each chapter runs on its own).

In [ ]:
def dtd_exp(tau, t_d=0.15, tau_ia=1.5):
    return np.where(tau >= t_d, np.exp(-(tau - t_d) / tau_ia) / tau_ia, 0.0)


def abundances(z, hydrogen=X_H):
    """[Fe/H], [Si/Fe], [Mg/Fe] from the mass fractions z[el] of the gas, on the Lodders (2003) scale."""
    with np.errstate(divide='ignore', invalid='ignore'):
        logn = {el: np.log10(z[el] / A[el]) for el in A}             # log number abundance (per unit mass)
        return dict(feh=logn['fe'] - np.log10(hydrogen) - (LOGEPS['fe'] - 12),
                    sife=logn['si'] - logn['fe'] - (LOGEPS['si'] - LOGEPS['fe']),
                    mgfe=logn['mg'] - logn['fe'] - (LOGEPS['mg'] - LOGEPS['fe']))


def one_zone_fe(t_end, dt, inflow, tau_star, eta, r=0.4, Mg0=0.0):
    """Chapter 2's model (Euler steps; CCSNe instantaneous, SNe Ia through the exponential DTD).
    inflow in Msun/Gyr. Returns t [Gyr], gas mass [Msun] and the abundances."""
    n = int(round(t_end / dt))
    t = np.arange(n + 1) * dt
    Mg = np.zeros(n + 1)
    Mg[0] = Mg0
    M = {el: np.zeros(n + 1) for el in YIELD}
    sfr = np.zeros(n + 1)
    f = dtd_exp(t) * dt
    for i in range(n):
        sfr[i] = Mg[i] / tau_star
        ia = np.dot(sfr[:i + 1], f[i::-1])
        loss = (1 + eta - r) * sfr[i]
        for el, (y_cc, y_ia) in YIELD.items():
            z = M[el][i] / Mg[i] if Mg[i] > 0 else 0.0
            M[el][i + 1] = M[el][i] + dt * (y_cc * sfr[i] + y_ia * ia - loss * z)
        Mg[i + 1] = Mg[i] + dt * (inflow(t[i]) - loss)
    with np.errstate(divide='ignore', invalid='ignore'):
        return dict(time=t, mgas=Mg, **abundances({el: M[el] / Mg for el in M}))

## 3.2 VICE with the same physics

`vice.singlezone` takes the same ingredients, under its own names:

| Chapter 2 | VICE | Note |
|---|---|---|
| inflow $\dot M_{\rm in}(t)$ | `mode='ifr'`, `func` | VICE wants $M_\odot\,$yr$^{-1}$: divide our $M_\odot\,$Gyr$^{-1}$ by $10^9$ |
| $\tau_\star$, $\eta$, $M_{g,0}$ | `tau_star`, `eta`, `Mg0` | $\tau_\star$ may be a function of time |
| instantaneous recycling, $r$ = 0.4 | `recycling=0.4` | a number means instantaneous |
| exponential DTD, 0.15 and 1.5 Gyr | `RIa='exp'`, `delay=0.15`, `tau_ia=1.5` | |
| constant yields | `vice.yields.ccsne.settings[el] = y` | the same for `sneia`; AGB yields set to zero |

VICE writes its output to a folder and cannot handle spaces in its path, so we run it inside a temporary folder.

In [ ]:
def run_vice(elements, t_end=10.0, dt=0.01, **kw):
    """Run vice.singlezone in a temporary folder; return its history (t = 0 ... t_end) as a dict of numpy arrays."""
    here = os.getcwd()
    with tempfile.TemporaryDirectory() as tmp:
        os.chdir(tmp)
        try:
            sz = vice.singlezone(name='run', elements=elements, dt=dt, **kw)
            h = sz.run(np.arange(round(t_end / dt) + 1) * dt, overwrite=True, capture=True).history
            keep = np.asarray(h['time']) <= t_end + 1e-6 * dt      # VICE writes one step past t_end: drop it
            return {k: np.asarray(h[k])[keep] for k in h.keys()}
        finally:
            os.chdir(here)


def set_constant_yields():
    for el, (y_cc, y_ia) in YIELD.items():
        vice.yields.ccsne.settings[el] = y_cc
        vice.yields.sneia.settings[el] = y_ia
        vice.yields.agb.settings[el] = lambda m, z: 0.0      # no AGB stars


FID = dict(tau_star=2.0, eta=2.5, Mg0=1e9)                   # the fiducial galaxy: inflow 1e9 Msun/Gyr
set_constant_yields()
h = run_vice(('fe', 'mg', 'si'), mode='ifr', func=lambda t: 1.0, recycling=0.4, RIa='exp', delay=0.15, tau_ia=1.5,
             smoothing=0.0, Zin=0.0, **FID)
matched = dict(time=h['time'], mgas=h['mgas'], **abundances({el: h[f'z({el})'] for el in A}))
npy = one_zone_fe(10.0, 0.01, lambda t: 1e9, **FID)

late = matched['time'] > 0.2                                  # skip t < 0.2 Gyr, where [Fe/H] -> -infinity
d_gas = np.max(np.abs(npy['mgas'] / matched['mgas'] - 1))
d_feh = np.max(np.abs(npy['feh'] - matched['feh'])[late])
d_sife = np.max(np.abs(npy['sife'] - matched['sife'])[late])
print(f'numpy vs VICE (same physics, dt = 10 Myr): gas {d_gas:.1e} (relative), [Fe/H] {d_feh:.4f} dex, [Si/Fe] {d_sife:.4f} dex')
assert d_gas < 1e-6 and d_feh < 0.002 and d_sife < 0.002   # numpy model = matched VICE

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(matched['time'], matched['mgas'] / 1e9, color='k', lw=3, label='VICE')
axes[0].plot(npy['time'], npy['mgas'] / 1e9, color='#eb6834', lw=1.5, ls='--', label='numpy (Ch. 2)')
axes[0].set(xlabel='time [Gyr]', ylabel=r'gas mass [$10^9\,M_\odot$]')
axes[1].plot(matched['feh'], matched['sife'], color='k', lw=3, label='VICE')
axes[1].plot(npy['feh'], npy['sife'], color='#eb6834', lw=1.5, ls='--', label='numpy (Ch. 2)')
axes[1].set(xlabel='[Fe/H]', ylabel='[Si/Fe]', xlim=(-2.5, 0.1))
axes[2].plot(matched['time'][late], (npy['feh'] - matched['feh'])[late], color='#2a78d6', lw=2, label='[Fe/H]')
axes[2].plot(matched['time'][late], (npy['sife'] - matched['sife'])[late], color='#1baf7a', lw=2, label='[Si/Fe]')
axes[2].set(xlabel='time [Gyr]', ylabel='numpy − VICE [dex]', ylim=(-0.003, 0.003))
for ax in axes:
    ax.legend(frameon=False, fontsize=9)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 3.1.** The fiducial galaxy (inflow $10^9\,M_\odot\,$Gyr$^{-1}$, initial gas $10^9\,M_\odot$,
$\tau_\star$ = 2 Gyr, $\eta$ = 2.5) with Chapter 2's physics, computed by VICE (black) and by our numpy model (orange
dashed), both with 10-Myr steps. Left: gas mass against time. Middle: [Si/Fe] against [Fe/H]. Right: numpy minus
VICE for [Fe/H] (blue) and [Si/Fe] (green), in dex, after 0.2 Gyr.

The two codes agree to $10^{-7}$ in gas mass and to better than 0.001 dex in the abundances: VICE integrates the same
equations with the same kind of time step. Our model is validated against an independent code, and VICE is
validated against equations we derived ourselves. Now we can trust VICE where our model cannot follow.

## 3.3 Switching on the missing physics, one item at a time

Our numpy model made four simplifications. VICE can drop each of them:
1. **Continuous recycling.** Stars return their gas when they die: massive stars within Myr, low-mass stars over
   Gyr. VICE computes this from the initial mass function and stellar lifetimes (`recycling='continuous'`).
2. **Metallicity-dependent yields.** The yields of Figure 2.1 change with the stars' metallicity. VICE accepts any
   function of $Z$ as a yield.
3. **AGB stars.** Stars of 1–8 $M_\odot$ return C, N and small amounts of other elements after 0.03–10 Gyr.
4. **Stellar lifetimes** follow a mass–lifetime relation; we use Padovani & Matteucci (1993), as H21 did.

Each step below adds one item to the previous one; the inflow, $\tau_\star$, $\eta$ and DTD stay fixed.

In [ ]:
def yield_of_Z(key):
    """A CCSN or SN Ia yield as a function of metallicity, from the H21 table (upper-bin lookup, as flexCE)."""
    table, zg = Y[key], Y['z']
    return lambda Z: float(table[min(len(zg) - 1, int(np.searchsorted(zg, Z, side='right')))])


def agb_yield(key):
    """AGB yield as a function of progenitor mass m and metallicity Z, divided by m (VICE's convention)."""
    table, zg, mass = Y[key], Y['z'], Y['agb_mass']
    def f(m, Z):
        if m < mass[0] or m > 8:
            return 0.0
        row = table[min(len(zg) - 1, int(np.searchsorted(zg, Z, side='right')))]
        return float(np.interp(m, mass, row) / m)
    return f


vice.mlr.setting = 'pm1993'                                  # Padovani & Matteucci (1993) lifetimes
steps = {}
common = dict(mode='ifr', func=lambda t: 1.0, RIa='exp', delay=0.15, tau_ia=1.5, smoothing=0.0, Zin=0.0, **FID)
set_constant_yields()
steps['0 Chapter 2 physics'] = run_vice(('fe', 'mg', 'si'), recycling=0.4, **common)
steps['1 + continuous recycling'] = run_vice(('fe', 'mg', 'si'), recycling='continuous', **common)
for el in A:
    vice.yields.ccsne.settings[el] = yield_of_Z(f'cc_{el}')
    vice.yields.sneia.settings[el] = yield_of_Z(f'ia_{el}')
steps['2 + Z-dependent yields'] = run_vice(('fe', 'mg', 'si'), recycling='continuous', **common)
for el in A:
    vice.yields.agb.settings[el] = agb_yield(f'agb_{el}')
steps['3 + AGB stars'] = run_vice(('fe', 'mg', 'si'), recycling='continuous', **common)

res = {}
for name, h in steps.items():
    ab = abundances({el: h[f'z({el})'] for el in A})
    rise = ab['feh'][1:] > np.maximum.accumulate(ab['feh'][1:]) - 1e-9          # monotonic part, for interpolation
    res[name] = dict(feh_end=ab['feh'][-1], sife_end=ab['sife'][-1], gas_end=h['mgas'][-1],
                     sife_m1=np.interp(-1.0, ab['feh'][1:][rise], ab['sife'][1:][rise]), ab=ab, t=h['time'])
print(f"{'step':28s} {'[Fe/H] 10 Gyr':>14s} {'[Si/Fe] 10 Gyr':>15s} {'[Si/Fe] at -1':>14s} {'gas 10 Gyr':>11s}")
for name, r in res.items():
    print(f"{name:28s} {r['feh_end']:14.3f} {r['sife_end']:15.3f} {r['sife_m1']:14.3f} {r['gas_end']:11.3e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cols = ('k', '#2a78d6', '#1baf7a', '#eb6834')
for (name, r), col in zip(res.items(), cols):
    axes[0].plot(r['t'], r['ab']['feh'], color=col, lw=2, label=name)
    axes[1].plot(r['ab']['feh'], r['ab']['sife'], color=col, lw=2, label=name)
axes[0].set(xlabel='time [Gyr]', ylabel='[Fe/H]', ylim=(-2.5, 0.3))
axes[1].set(xlabel='[Fe/H]', ylabel='[Si/Fe]', xlim=(-2.5, 0.3))
for ax in axes:
    ax.legend(frameon=False, fontsize=8)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

**Figure 3.2.** The fiducial galaxy as the approximations are removed one by one (each step keeps the previous
ones): Chapter 2 physics (black), plus continuous recycling (blue), plus metallicity-dependent yields (green), plus
AGB stars and Padovani & Matteucci lifetimes (orange). Left: [Fe/H] against time; right: [Si/Fe] against [Fe/H].

The table above gives the numbers. For this galaxy:
- **Continuous recycling** changes almost nothing (0.004 dex in [Fe/H], 1% in gas mass). With a steady inflow, the
  gas returned by old stars is a small addition to the inflow, and $r = 0.4$ is close to what a Kroupa population
  returns over 10 Gyr.
- **Metallicity-dependent yields** are the largest effect, but only late and at low metallicity: [Si/Fe] at 10 Gyr
  rises by 0.05 dex, because the CCSN Si yield grows by 20% between $Z$ = 0.002 and 0.014 (Figure 2.1), and the
  plateau below [Fe/H] ≈ −1.2 rises by up to 0.02 dex.
- **AGB stars** change nothing visible: they make almost no Si or Fe.

The shape of the track, plateau, knee and decline, survives every step: for these elements Chapters 1 and 2
captured the physics that matters. Do not generalise this to every element: AGB stars dominate C, N and the
s-process elements, and continuous recycling matters more when the inflow stops.

In [ ]:
r0, r1 = res['0 Chapter 2 physics'], res['1 + continuous recycling']
r2, r3 = res['2 + Z-dependent yields'], res['3 + AGB stars']
pairs = ((r0, r1), (r1, r2), (r2, r3))
d_feh_end = [b['feh_end'] - a['feh_end'] for a, b in pairs]
d_sife_end = [b['sife_end'] - a['sife_end'] for a, b in pairs]
print('step 1, 2, 3 shifts of [Fe/H] at 10 Gyr: ', np.round(d_feh_end, 3))
print('step 1, 2, 3 shifts of [Si/Fe] at 10 Gyr:', np.round(d_sife_end, 3))
# regression (development run, 2026-11-12): the quoted shifts
np.testing.assert_allclose(d_feh_end, [0.004, -0.007, 0.000], atol=0.002)
np.testing.assert_allclose(d_sife_end, [0.001, 0.046, 0.001], atol=0.003)

## 3.4 Bursts

Chapter 0 showed the LMC's [Mg/Fe] *rising* above [Fe/H] ≈ −0.8. A steady galaxy cannot do this: once SNe Ia have
started, [α/Fe] only falls. A **burst** can: if star formation suddenly increases, CCSNe add α-elements at once,
while the SNe Ia from the burst arrive later. H21 model bursts as a temporary rise of the star-formation efficiency,

$${\rm SFE}(t) = 1/\tau_\star(t) = {\rm SFE}_0\left[1 + (f_b - 1)\,e^{-(t-\tau_b)^2/2\sigma_b^2}\right],$$

a Gaussian of height $f_b$ centred at time $\tau_b$ with width $\sigma_b$. H21's inflow is a "delayed-τ" law,
$\dot M_{\rm in} = (M_i/\tau_i)(t/\tau_i)\,e^{-t/\tau_i}$, with $M_i = 6\times10^{10}\,M_\odot$ and initial gas
$M_0 = 3\times10^9\,M_\odot$ for every galaxy. From here on we use H21's full setup (continuous recycling, their
yields for 11 elements, AGB stars), and compute hydrogen as 1 − (helium + metals) rather than holding it fixed.

In [ ]:
ELEMENTS = ('he', 'c', 'n', 'o', 'ne', 'mg', 'si', 's', 'ca', 'fe', 'ni')
M_GAS_0, M_INFLOW = 3e9, 6e10
for el in ELEMENTS:                                           # H21's yields for all 11 elements
    vice.yields.ccsne.settings[el] = yield_of_Z(f'cc_{el}')
    vice.yields.sneia.settings[el] = yield_of_Z(f'ia_{el}')
    vice.yields.agb.settings[el] = agb_yield(f'agb_{el}')


def sfe(t, p):
    """H21's star-formation efficiency [Gyr^-1] with a Gaussian burst (f_b > 1) or dip (f_b < 1)."""
    if p['tau_b'] is None:
        return p['sfe0']
    return p['sfe0'] * (1 + (p['f_b'] - 1) * np.exp(-0.5 * ((t - p['tau_b']) / p['sigma_b']) ** 2))


def run_h21(p, dt=0.03, t_end=12.0):
    """One H21 model in VICE. Returns time [Gyr], SFR [Msun/yr], gas [Msun], [Fe/H], [Si/Fe], [Mg/Fe]."""
    h = run_vice(ELEMENTS, t_end=t_end, dt=dt, mode='ifr', Mg0=M_GAS_0, eta=p['eta'],
                 func=lambda t: M_INFLOW / p['tau_i'] * (t / p['tau_i']) * np.exp(-t / p['tau_i']) / 1e9,
                 tau_star=lambda t: 1 / sfe(t, p), IMF='kroupa', m_lower=0.1, m_upper=100.0,
                 RIa='exp', delay=0.15, tau_ia=1.5, recycling='continuous', postMS=0.0, Z_solar=0.014,
                 schmidt=False, smoothing=0.0, Zin=0.0, enhancement=1.0)
    z = {el: h[f'z({el})'] for el in ELEMENTS}
    hydrogen = 1 - sum(z.values())                            # H = 1 - He - metals (mass fractions)
    return dict(time=h['time'], sfr=h['sfr'], mgas=h['mgas'], **abundances(z, hydrogen))


lmc = dict(sfe0=0.015, eta=2.0, tau_i=2.0, tau_b=11.0, sigma_b=0.75, f_b=6.0)       # H21's LMC (Table 4)
burst = run_h21(lmc)
steady = run_h21({**lmc, 'f_b': 1.0})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for m, col, lab in ((steady, '#888888', 'no burst'), (burst, '#2a78d6', 'burst: $f_b$ = 6 at 11 Gyr')):
    axes[0].plot(m['time'], m['sfr'], color=col, lw=2, label=lab)
    axes[1].plot(m['feh'], m['sife'], color=col, lw=2, label=lab)
axes[0].set(xlabel='time [Gyr]', ylabel=r'SFR [$M_\odot\,$yr$^{-1}$]')
axes[1].set(xlabel='[Fe/H]', ylabel='[Si/Fe]', xlim=(-2.5, 0.0))
for ax in axes:
    ax.legend(frameon=False)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()

def largest_rise(x):
    """Largest increase of x above its running minimum (0 for a sequence that never rises)."""
    return np.max(x - np.minimum.accumulate(x))


after = burst['time'] > 1.0
rise = {x: largest_rise(burst[x][after]) for x in ('sife', 'mgfe')}
print(f"no burst: largest rise after 1 Gyr: [Si/Fe] {largest_rise(steady['sife'][after]):.4f}, "
      f"[Mg/Fe] {largest_rise(steady['mgfe'][after]):.4f} dex")
print(f"burst:    largest rise after 1 Gyr: [Si/Fe] {rise['sife']:.3f}, [Mg/Fe] {rise['mgfe']:.3f} dex")
assert largest_rise(steady['sife'][after]) < 1e-6 and largest_rise(steady['mgfe'][after]) < 1e-6   # monotonic
assert rise['mgfe'] > rise['sife'] > 0.01                          # burst: both climb back, Mg more than Si
sfr_before, sfr_peak = burst['sfr'][np.argmin(np.abs(burst['time'] - 8.5))], burst['sfr'].max()
print(f'SFR at 8.5 Gyr {sfr_before:.2f}, peak {sfr_peak:.2f} Msun/yr')
assert abs(sfr_before - 0.7) < 0.05 and abs(sfr_peak - 3.5) < 0.1 and abs(rise['sife'] - 0.04) < 0.005

**Figure 3.3.** H21's best LMC model (Table 4: SFE$_0$ = 0.015 Gyr$^{-1}$, $\eta$ = 2, $\tau_i$ = 2 Gyr, burst
$f_b$ = 6 at $\tau_b$ = 11 Gyr with $\sigma_b$ = 0.75 Gyr; blue) and the same model without the burst (grey). Left:
star-formation rate against time; right: [Si/Fe] against [Fe/H].

Without the burst, [Si/Fe] declines monotonically. The burst raises the star-formation rate five-fold, from 0.7 to
3.5 $M_\odot\,$yr$^{-1}$, around 11 Gyr; fresh CCSNe push [Si/Fe] back up by 0.04 dex and [Mg/Fe] by 0.045 dex
(printed above). In the data (Chapter 0) the LMC's medians rise by 0.04 dex in [Si/Fe] and 0.08 dex in [Mg/Fe]
above [Fe/H] ≈ −0.8: the model's burst matches the silicon, which it was fitted to, and undershoots the magnesium.
Note how *late* this burst is: in the model, the LMC's most metal-rich stars formed in the last 2 Gyr.

## 3.5 [Fe/H] is not a clock

H21's best-fit models of the five galaxies (their Table 4) differ in efficiency, outflow, inflow time scale and
bursts. Running them shows how differently they enrich.

In [ ]:
H21_BEST = {
    'LMC':    dict(sfe0=0.015, eta=2.0,   tau_i=2.0, tau_b=11.0, sigma_b=0.75, f_b=6.0),
    'SMC':    dict(sfe0=0.008, eta=10.0,  tau_i=2.0, tau_b=7.0,  sigma_b=0.75, f_b=4.0),
    'Sgr':    dict(sfe0=0.03,  eta=17.5,  tau_i=1.0, tau_b=5.0,  sigma_b=0.5,  f_b=0.01),
    'GSE':    dict(sfe0=0.14,  eta=6.0,   tau_i=2.5, tau_b=None, sigma_b=None, f_b=1.0),
    'Fornax': dict(sfe0=0.03,  eta=100.0, tau_i=1.0, tau_b=6.0,  sigma_b=5.0,  f_b=0.01),
}
COLORS = {'LMC': '#2a78d6', 'SMC': '#eb6834', 'Sgr': '#1baf7a', 'GSE': '#eda100', 'Fornax': '#e87ba4'}
models = {g: run_h21(p) for g, p in H21_BEST.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
t_m1 = {}
for g, m in models.items():
    axes[0].plot(m['time'], m['feh'], color=COLORS[g], lw=2, label=g)
    axes[1].plot(m['feh'], m['sife'], color=COLORS[g], lw=2, label=g)
    t_m1[g] = m['time'][np.argmax(m['feh'] > -1.0)]
axes[0].axhline(-1, color='0.5', lw=0.8, ls=':')
axes[0].set(xlabel='time [Gyr]', ylabel='[Fe/H]', ylim=(-2.5, 0.1))
axes[1].set(xlabel='[Fe/H]', ylabel='[Si/Fe]', xlim=(-2.5, 0.1))
for ax in axes:
    ax.legend(frameon=False, fontsize=9)
    ax.grid(color='0.9', lw=0.6)
fig.tight_layout()
print('time to reach [Fe/H] = -1 [Gyr]:', {g: round(float(v), 2) for g, v in t_m1.items()})
assert max(t_m1.values()) / min(t_m1.values()) > 3            # not a common clock
np.testing.assert_allclose([t_m1[g] for g in H21_BEST], [6.48, 7.26, 3.84, 1.20, 11.34], atol=0.031)   # quoted below

# The same five models are part of the precomputed grid used in Chapter 4: check that we computed the same thing.
G = np.load(fetch('h21_grid_vice.npz'))
for g, p in H21_BEST.items():
    sel = G['galaxy'] == g
    for k in ('sfe0', 'eta', 'tau_b', 'sigma_b', 'f_b'):
        sel &= np.isnan(G[k]) if p[k] is None else np.isclose(G[k], p[k])
    (j,) = np.flatnonzero(sel)
    assert np.allclose(G['time'], models[g]['time'])
    ok = models[g]['time'] > 0.1
    assert np.max(np.abs(G['feh'][j] - models[g]['feh'])[ok]) < 1e-4    # regression: matches the shipped grid
    assert np.max(np.abs(G['sife'][j] - models[g]['sife'])[ok]) < 1e-4
print('the five models match the precomputed grid of Chapter 4 to < 1e-4 dex')

**Figure 3.4.** H21's best-fit models of the five galaxies (Table 4), run with VICE. Left: [Fe/H] against time
(dotted: [Fe/H] = −1); right: [Si/Fe] against [Fe/H]. Colours as in Chapter 0.

The galaxies reach [Fe/H] = −1 at very different times (printed above): GSE, with the highest efficiency, after
1.2 Gyr; Sgr after 3.8 Gyr; the LMC and SMC after 6.5–7.3 Gyr; Fornax, whose huge outflow ($\eta$ = 100) keeps its
gas metal-poor, only after 11.3 Gyr. Two stars at the same [Fe/H] in two of these galaxies can differ in age by
10 Gyr. The dips in the Sgr and Fornax tracks are H21's "negative bursts" ($f_b$ = 0.01): star formation nearly
stops, SNe Ia from earlier stars keep adding iron, and [Si/Fe] falls steeply until star formation resumes. The five models also agree
with the precomputed grid that Chapter 4 downloads, so Chapter 4 can fit with those grids without rebuilding VICE.

## Summary

- With the same physics, VICE and our numpy model agree to $10^{-7}$ in gas mass and 0.001 dex in abundances.
- Removing the approximations of Chapters 1–2 changes [Fe/H] and [Si/Fe] by less than 0.05 dex for this galaxy;
  metallicity-dependent yields matter most, continuous recycling and AGB stars hardly at all.
- A burst in the star-formation efficiency makes [α/Fe] rise, as in the LMC.
- [Fe/H] is not a clock: H21's five models reach [Fe/H] = −1 between 1.2 and 11.3 Gyr.

## Exercises

1. Rerun Figure 3.2 with `RIa='plaw'` (VICE's power-law DTD). Which step now dominates?
2. Move the LMC burst to $\tau_b$ = 6 Gyr. Where along the [Si/Fe]–[Fe/H] track does the upturn appear now?
3. Plot the gas mass of the five H21 models against time. Which galaxies end with more gas than stars?